# Tutorial 7 — Neutron beams and spectra

**Goal:** go beyond the default thermal beam: cold monochromatic and polychromatic
spectra, and how they change neutron contrast.

**You will learn:** `NEUTRON_MODES`, `cold_mono_beam`, `cold_poly_beam`, `mu_n_lut_for_beam`,
and how to put beam-specific attenuation into a phantom.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

## 1. Beams

A `NeutronBeam` is an energy grid plus normalised flux weights. The default simulation uses a
monochromatic **thermal** beam (25.3 meV, 1.8 Å). Cold beams have longer wavelengths, where
**absorption grows as $1/v$** — e.g. lithium, boron or cobalt become much more visible.

In [ ]:
for name, beam in nxs.NEUTRON_MODES.items():
    print(f"{name:>10}: {beam}")
fig = nxs.plot_spectra()

## 2. Attenuation depends on the beam

`mu_n_lut_for_beam` returns the effective $\mu_n$ of each material for a given beam
(flux-weighted for polychromatic beams, thin-sample limit).

In [ ]:
mats = [nxs.MATERIALS[k] for k in ["water", "hdpe", "aluminum", "iron", "lithium", "lco", "nmc811"]]
beams = {"thermal": nxs.NEUTRON_MODES["thermal"], "cold 5 meV": nxs.cold_mono_beam(5.0),
         "cold 2 meV": nxs.cold_mono_beam(2.0), "ILL-NeXT": nxs.ill_next_beam()}
luts = {name: nxs.mu_n_lut_for_beam(mats, beam) for name, beam in beams.items()}
x = np.arange(len(mats))
fig, ax = plt.subplots(figsize=(9, 3.5))
for i, (name, lut) in enumerate(luts.items()):
    ax.bar(x + i * 0.2, lut, width=0.2, label=name)
ax.set_xticks(x + 0.3, [m.symbol for m in mats]); ax.set_ylabel(r"$\mu_n$ [cm$^{-1}$]"); ax.legend()

## 3. Simulating with a cold beam

The projector reads the neutron attenuation from the phantom's volumes, so we can replace
them with beam-specific values. `set_phantom_neutron_mu` writes one $\mu_n$ per label (all in
the absorption channel).

In [ ]:
from neutron_xray_sim.physics.ncrystal_bragg import set_phantom_neutron_mu

def neutron_slice(beam, N=48):
    ph = nxs.make_phantom("battery", N=N)
    set_phantom_neutron_mu(ph, nxs.mu_n_lut_for_beam(ph.materials, beam))
    _, n = nxs.make_sinogram_pair(ph, n_angles=90, verbose=False)
    return nxs.reconstruct(n)[N // 2]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, beam) in zip(axes, [("thermal", beams["thermal"]), ("cold 2 meV", beams["cold 2 meV"])]):
    im = ax.imshow(neutron_slice(beam), cmap="gray"); ax.set_title(name); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, label=r"$\mu_n$ [cm$^{-1}$]")

## 4. Bragg edges (optional)

Crystalline materials show sharp **Bragg edges** in their wavelength-dependent cross section,
which lets energy-resolved neutron imaging distinguish crystal phases (e.g. ferrite vs.
austenite). DIANA computes these with **NCrystal** (`pip install ncrystal`); see
`notebooks/05_bragg_edge_corrosion_phantom.ipynb` for a full study.

In [ ]:
from neutron_xray_sim.physics import ncrystal_bragg as nb
if nb.NCRYSTAL_AVAILABLE:
    wl = np.linspace(1.5, 6.0, 300)
    for phase in ["martensite", "austenite"]:
        plt.plot(wl, nb.load_phase(phase).mu_total_cm(wl), label=phase)
    plt.xlabel("wavelength [Å]"); plt.ylabel(r"$\mu$ [cm$^{-1}$]"); plt.legend()
else:
    print("NCrystal is not installed — skipping (pip install ncrystal).")

**Exercise:** lithium-ion battery materials contain Li and Co, both strong absorbers.
Which beam in `NEUTRON_MODES` maximises the contrast between `lco` and `graphite`?